# Unit 12 exercises: tuning against race time

A search can make an agent faster on the tracks it is scored on and worse everywhere else.
These exercises measure the objective's noise, map a few knobs, run a small evolution strategy and a random search yourself, check a training win on the validation track, apply the crash-aware check and the margin rule, profile a race, and walk into the rounding traps on purpose.
The Challenges search the speed loop's gains and top speed, and race a policy gradient against coordinate search on the same budget.
Every number here is a number about racekit's **carla-like car**: a model, not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Results from one exercise are reused in later ones, so run the exercises in order.
Every race goes through the setup's `race()` helper: `agents/unit-11.py` unless a `source` is given, the constants in `setting` changed, **the timing off** (`MPC_BUDGET_MS = None`), the track's own line and measured grip written in, on `carla_like(noise=True)` (car `k` of Unit 11's perturbed family if `k` is given), **1 lap** unless an exercise says otherwise.
A 1-lap race takes about 2 s, a 3-lap race about 6; keep to the races each exercise asks for.
**Never race `test-circuit-seed2028`** here: it is the unit's test track, raced once, by the milestone.

In [ ]:
import cProfile
import math
import pstats
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_11, UNIT_12 = AGENTS / "unit-11.py", AGENTS / "unit-12.py"
workdir = Path(mkdtemp(prefix="roar-unit-12-ex-"))
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
VALIDATION = "test-circuit-seed2026"  # the unit's validation track
# Unit 8's grip probes (m/s^2 per grip section, rounded to 0.01); test-circuit's ship in the agents.
MEASURED_MU_G = {
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
}


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# The lines: test-circuit's as the agents ship it, the others solved here.
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
control, line_offsets = {}, {}
for name in ("test-circuit", "test-circuit-seed1", VALIDATION):
    track = load_track(name)
    if name == "test-circuit":
        line_offsets[name] = np.array(Unit11(track.waypoints, None).offsets)
    else:
        control[name] = solve_line(track)
        line_offsets[name] = spline_basis(len(track)) @ control[name]


written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def on_track(track_name, **constants):
    # The constants a race on this track needs: its own line and measured grip (test-circuit: as shipped).
    if track_name != "test-circuit":
        constants = {"LINE_CONTROL": tuple(float(value) for value in control[track_name]),
                     "MU_G": MEASURED_MU_G[track_name], **constants}
    return constants


def race(setting, track="test-circuit", seed=0, laps=1, source=UNIT_11, k=None, max_seconds=450.0):
    # One race of `source` with the constants in `setting` changed and the timing off, on `track` with its line
    # and grip, on the carla-like car with its noise (car k of Unit 11's perturbed family if k is given).
    path = variant(source, **on_track(track, **setting), **NO_TIMING)
    return run_race(path, track, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=max_seconds)


def line_error(result):
    # The largest |cross-track error from the race's own line| over the driving rows (held == 0).
    offsets = line_offsets[result.track]
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result):
    return result.finished and result.collisions == 0 and line_error(result) <= CLEAN_ERROR


def same_race(a, b):
    # The same race: every telemetry row equal, not only the time.
    return len(a.telemetry) == len(b.telemetry) and all(
        np.sum(a.telemetry[name] != b.telemetry[name]) == 0 for name in a.telemetry.dtype.names)

## Exercise 1 - The objective's noise

How much does a race time move when nothing changes but the race seed?
Race `agents/unit-11.py` (timing off: `variant(UNIT_11, **NO_TIMING)`) five times on `test-circuit`, 1 lap, race seeds 0 to 4, with `repeat_races(source, "test-circuit", n=5, seeds=(0, 1, 2, 3, 4), car_factory=lambda: carla_like(noise=True), laps=1)`.
Later exercises compare against these five races; keep them.

The cell also gives the probe record's per-seed **five-track sums** of the same file (the lesson's objective, seeds 0 to 9).

Define:

- `noise_runs`: the `RepeatResult`; `one_lap_times`: its times as a list of floats; `one_lap_spread`: its spread (s);
- `shipped_by_seed`: a dict from race seed to that seed's race result (`noise_runs.results`, in seed order);
- `ticks`: each time as a whole number of ticks (`round(t / DT)`), and `on_ticks`: `True` when every time is within 1e-9 s of its ticks × `DT`;
- `sum_spread_1lap`, `sum_spread_3lap`: the spreads of `SUMS_ONE_LAP` and `SUMS_THREE_LAPS`, and `sum_to_track`: `sum_spread_1lap / one_lap_spread`;
- `fig`, `ax`: the five times against the race seed, with their median as a horizontal line, axis labels and a title;
- `why_sums`: two sentences: why is the five-track sum's spread bigger than one track's, and why does the lesson's search compare **per-seed sums** at two seeds by Unit 1's rule instead of one race against another?

In [ ]:
# The probe record of plan roar-012 (p1): agents/unit-11.py, five training tracks, the sum per race seed (0-9), s.
SUMS_ONE_LAP = [476.80, 476.70, 477.00, 476.70, 476.60, 476.95, 476.55, 476.90, 476.65, 476.95]
SUMS_THREE_LAPS = [1383.25, 1383.05, 1383.25, 1382.85, 1382.95, 1383.45, 1382.75, 1383.15, 1383.15, 1383.55]

noise_runs, one_lap_times, one_lap_spread = None, None, None  # TODO
shipped_by_seed = {}  # TODO
ticks, on_ticks = None, None  # TODO
sum_spread_1lap, sum_spread_3lap, sum_to_track = None, None, None  # TODO
fig, ax = None, None  # TODO
why_sums = "TODO"

## Exercise 2 - A knob map of three knobs

Classify three constants of `agents/unit-11.py` by racing them: `TARGET_SPEED = 50.0`, `LEAD_MIN_M = 8.0` and `K_E = 0.5`.
Race each on `test-circuit`, 1 lap, race seeds 0 and 1 (`race({knob: value}, seed=s)`): six races.
Compare each with Exercise 1's races of the shipped file at the same two seeds.

Define:

- `KNOBS`: given; `knob_races`: a dict from knob name to its list of two results (seeds 0, 1), and `knob_times`: the same with the race times;
- `shipped_two`: the shipped file's times at seeds 0 and 1 (from Exercise 1);
- `deltas`: a dict from knob name to the median of its two times minus the median of `shipped_two` (s);
- `beyond_noise`: a dict from knob name to `True` when Unit 1's rule says the two differ, either way (`noise_aware_better` in either order);
- `identical`: a dict from knob name to `True` when both its races are the same race (`same_race`) as the shipped race at the same seed;
- `verdict`: a dict from knob name to `"dead"` (identical), `"acts, beyond the noise"` or `"acts, within the noise"`;
- `k_e_reason`: one sentence: which part of the agent reads `K_E`, and why the shipped file never runs it;
- `sweep_limits`: two sentences: what can a two-seed, one-track, one-lap comparison miss?
  Use the lesson's findings on 1 lap against 3, and on five tracks against one.

In [ ]:
KNOBS = {"TARGET_SPEED": 50.0, "LEAD_MIN_M": 8.0, "K_E": 0.5}

knob_races, knob_times = {}, {}  # TODO
shipped_two = None  # TODO
deltas = {}  # TODO
beyond_noise = {}  # TODO
identical = {}  # TODO
verdict = {}  # TODO
k_e_reason = "TODO"
sweep_limits = "TODO"

## Exercise 3 - A (1+2) evolution strategy, live

Run the lesson's evolution strategy at a size you can watch.

- **Knobs and box:** `MU_FRACTION` in [0.48, 0.66] and `A_DEC` in [4.0, 8.0], starting from `unit-11.py`'s 0.54 and 5.0.
- **Objective:** the sum of the 1-lap race times on the two tracks of `ES_TRACKS`, race seed 0.
  A setting is **rejected** if either of its two races is not clean.
- **Acceptance:** one race seed gives no spread, so Unit 1's rule cannot be used; instead a child replaces the parent when it is clean and its sum is lower than the parent's by more than `NOISE_BAND` (0.30 s, the hook's noise).
- **Children:** each generation draws two children from `rng = np.random.default_rng(3)`: for the first child, `MU_FRACTION` then `A_DEC`, then the same for the second child; each knob is the parent's value plus `rng.normal(0.0, sigma[knob])`, clipped to its box and rounded with `round(x, 2)`.
- **Step size:** `sigma` starts at one grid step (0.02 and 0.5); if the better clean child is accepted it becomes the parent and `sigma` is multiplied by 1.5, otherwise by 0.8.
- **Budget:** the parent's evaluation plus **two** generations: 10 races.

Write `evaluate(setting, tracks=ES_TRACKS, seed=0, source=UNIT_11)`: race the setting on each track with `max_seconds=200.0` (a clean lap takes about 96 s, so a crash loop is cut short) and return a dict with `"setting"`, `"times"` (a list, in track order), `"sum"`, `"clean"` (all races clean) and `"results"`.
Exercises 4 and 9 reuse it.

Define:

- `evaluate`: the function; `start_eval`: `evaluate` of the starting setting;
- `es_log`: a list with one dict per generation: `"children"` (their evaluations), `"accepted"` (`True` or `False`), `"sigma"` (after the update);
- `parent`, `parent_eval`: the final parent setting and its evaluation; `es_races`: the number of races raced (10);
- `es_gain`: `start_eval["sum"] - parent_eval["sum"]` (s);
- `fig`, `ax`: every evaluation's sum against the race count after it (not-clean evaluations drawn with a different marker), and the parent's sum as a step line, with a legend, axis labels and a title;
- `es_honest`: two sentences: after 10 races, would you call the final parent better than `unit-11.py`, and what did the record's (1+4)-ES do with 200 three-lap races (the lesson's p3)?

In [ ]:
ES_TRACKS = ("test-circuit", "test-circuit-seed1")
ES_BOX = {"MU_FRACTION": (0.48, 0.66), "A_DEC": (4.0, 8.0)}
ES_START = {"MU_FRACTION": 0.54, "A_DEC": 5.0}
NOISE_BAND = 0.30  # s


def evaluate(setting, tracks=ES_TRACKS, seed=0, source=UNIT_11):
    return None  # TODO: {"setting", "times", "sum", "clean", "results"}


start_eval = None  # TODO
es_log = []  # TODO
parent, parent_eval, es_races = None, None, None  # TODO
es_gain = None  # TODO
fig, ax = None, None  # TODO
es_honest = "TODO"

## Exercise 4 - Random search, and why a not-clean race rejects

Draw four whole settings at random and evaluate each with Exercise 3's `evaluate` (two tracks, race seed 0): eight races.
Use `rng = np.random.default_rng(12)`; for each setting draw `MU_FRACTION` in [0.48, 0.66], then `A_DEC` in [4.0, 8.0], then `A_ACC` in [3.0, 6.0], each with `rng.uniform(low, high)` and rounded with `round(x, 2)`.
Compare every setting with Exercise 3's `start_eval` (`unit-11.py`).

Define:

- `draws`: the four settings (dicts), in the order drawn, and `random_evals`: their evaluations;
- `not_clean`: the settings whose evaluation is not clean;
- `faster_but_not_clean`: the settings that are not clean **and** whose sum is lower than `start_eval["sum"]`: the ones that would have been taken if a race that is not clean only counted its time;
- `best_random`: the clean setting with the lowest sum (`None` if no setting is clean), and `random_beats_start`: `True` when its sum is lower than `start_eval["sum"]` by more than `NOISE_BAND`;
- `fig`, `ax`: `MU_FRACTION` (x) against `A_ACC` (y) for the four draws, clean and not-clean settings with different markers, `unit-11.py`'s values marked too, with a legend, axis labels and a title;
- `why_random_is_slow`: two sentences: why did random search gain least of the three searches in the lesson's p3 record, and what does a not-clean but faster setting teach about the clean rule?

In [ ]:
RANDOM_BOX = {"MU_FRACTION": (0.48, 0.66), "A_DEC": (4.0, 8.0), "A_ACC": (3.0, 6.0)}

draws, random_evals = [], []  # TODO
not_clean = None  # TODO
faster_but_not_clean = None  # TODO
best_random, random_beats_start = None, None  # TODO
fig, ax = None, None  # TODO
why_random_is_slow = "TODO"

## Exercise 5 - A training win on the validation track

The lesson's live coordinate search ended its 40 races at `MU_FRACTION = 0.58` (`A_DEC` 5.0, `A_ACC` 4.0), 7.98 s faster than `unit-11.py` over five 1-lap training races (476.75 → 468.77 s).
Check it where it was never scored: race it on the validation track `test-circuit-seed2026`, **3 laps**, race seeds 0 and 1 (`race({"MU_FRACTION": 0.58}, VALIDATION, seed=s, laps=3)`): two races.
`unit-11.py`'s times there are given (the premise, U12.5a).

Define:

- `validation_races`: the two results; `validation_times`: their times; `validation_clean`: a list of `clean(result)`; `validation_max_error`: the largest `line_error` of the two (m);
- `beats_unit11`: `True` when both are clean and Unit 1's rule says they beat `UNIT_11_SEED2026` (the same seeds);
- `training_gain_per_lap`: (476.75 − 468.77) / 5, and `validation_gain_per_lap`: (the median of `UNIT_11_SEED2026` − the median of `validation_times`) / 3, both in seconds per lap;
- `still_a_test`: three sentences: you have now raced `MU_FRACTION = 0.58` on the validation track.
  If you ship it **because** of this result, can `test-circuit-seed2026` still tell you how it does on a track it has never seen?
  What would you race before shipping it (think of Unit 11's perturbed cars on this track, and the test track), and how many times would you race the test track?

In [ ]:
UNIT_11_SEED2026 = [282.05, 282.00]  # s: agents/unit-11.py on test-circuit-seed2026, 3 laps, race seeds 0 and 1

validation_races, validation_times = None, None  # TODO
validation_clean, validation_max_error = None, None  # TODO
beats_unit11 = None  # TODO
training_gain_per_lap, validation_gain_per_lap = None, None  # TODO
still_a_test = "TODO"

## Exercise 6 - The crash-aware check, and the margin rule

**Part A: the check.**
The crash-aware objective adds Unit 11's three lowest-grip training cars to every evaluation.
Race the speed search's setting (`SPEED_SETTING`, on `unit-11.py`) and `agents/unit-12.py` as shipped (`source=UNIT_12`, no constants changed) on `test-circuit` on each of cars 16, 18 and 8, race seed k mod 5, 1 lap, `max_seconds=250.0` (`race(setting, k=k, seed=k % 5, source=..., max_seconds=250.0)`): six races.

**Part B: the margin rule, no races.**
`CRASH_AWARE` is the lesson's table of the crash-aware search's ten evaluations (3 laps, 16 races each).
Unit 11's margin rule: find the **edge**, the highest value of a knob that is clean, and ship one grid step inside it.

Define:

- `low_grip_races`: a dict from `("speed", k)` and `("unit-12", k)` to the result, and `low_grip_clean`: the same with `clean(result)`;
- `speed_rejected`: `True` when any of the speed setting's three races is not clean (the crash-aware objective would reject it);
- `crash_costs`: a dict from each car on which the speed setting had a collision to its race time minus `unit-12.py`'s on the same car (s; a race cut at 250 s counts at 250 s);
- `clean_settings`: the settings in `CRASH_AWARE` with no race that was not clean; `best_clean`: the one with the lowest median of its two sums;
- `clean_neighbours`: the settings one grid step from `best_clean` on exactly one knob (steps 0.02, 0.5, 0.5) that are in `clean_settings`;
- `a_dec_edge`: the highest clean `A_DEC` with `MU_FRACTION` 0.54 and `A_ACC` 4.0, and `margin_setting`: the margin rule's setting along `A_DEC` (one grid step inside the edge, as a tuple); `margin_is_unit11`: `True` when it is `unit-11.py`'s (0.54, 5.0, 4.0);
- `which_to_race`: three sentences: the lesson measured that the edge (`unit-12.py`) is 1.35 to 1.40 s faster on the validation and test tracks and clean on 20 of Unit 11's 24 held-out cars, where `unit-11.py` is clean on 24.
  Which would you race, and why?
  What does your answer assume about how far the real car is from racekit's?

In [ ]:
SPEED_SETTING = {"MU_FRACTION": 0.60, "A_DEC": 5.5, "A_ACC": 4.5}  # the speed search's result, on the grid
LOW_GRIP_CARS = (16, 18, 8)  # Unit 11's three lowest-grip training cars
# The lesson's table (probe p4): (MU_FRACTION, A_DEC, A_ACC) -> ((the two per-replicate sums, s), what was not clean)
CRASH_AWARE = {
    (0.54, 5.0, 4.0): ((2224.90, 2224.35), ""),
    (0.56, 5.0, 4.0): ((2411.70, 2205.10), "car 8"),
    (0.52, 5.0, 4.0): ((2245.50, 2245.25), ""),
    (0.54, 5.5, 4.0): ((2215.35, 2215.25), ""),
    (0.54, 4.5, 4.0): ((2238.75, 2238.90), ""),
    (0.54, 6.0, 4.0): ((2984.10, 2966.65), "cars 16, 18, 8, nominal test-circuit"),
    (0.54, 5.5, 4.5): ((2718.85, 2203.70), "cars 16, 8"),
    (0.54, 5.5, 3.5): ((2246.25, 2246.45), "cars 18, 8"),
    (0.56, 5.5, 4.0): ((2495.40, 2501.40), "cars 8, 16"),
    (0.52, 5.5, 4.0): ((2235.50, 2235.65), "car 18"),
}

low_grip_races, low_grip_clean = {}, {}  # TODO
speed_rejected = None  # TODO
crash_costs = {}  # TODO
clean_settings, best_clean = None, None  # TODO
clean_neighbours = None  # TODO
a_dec_edge, margin_setting, margin_is_unit11 = None, None, None  # TODO
which_to_race = "TODO"

## Exercise 7 - Profile a shorter horizon

Profile one race of `agents/unit-12.py` with `MPC_HORIZON = 20` (half the shipped 40): `test-circuit`, 1 lap, race seed 0, timing off.
Write the variant with `variant(UNIT_12, MPC_HORIZON=20, **NO_TIMING)` (you need its file name), then `profiler = cProfile.Profile()`, `profiler.enable()`, `run_race(...)` on `carla_like(noise=True)`, `profiler.disable()`.
Read it with `pstats.Stats(profiler)`: print the agent's functions sorted by cumulative time (`print_stats(path.stem, 12)`), then everything sorted by own time (`print_stats(8)`).

Your times depend on the machine and its load: **print** them and read them; do not compare them with a fixed number.

Define:

- `horizon_path`: the variant's path; `profiler`; `horizon_race`: the race result; `stats`: the `pstats.Stats`;
- `step_p50_ms`, `step_p99_ms`: `step()`'s 50th and 99th percentiles in this race (`horizon_race.step_ms`; the profiler slows them);
- `where_time_goes`: two sentences: which of the agent's functions takes the most cumulative time, about what share of `step()` is it here, and how does that compare with the lesson's p6 share at horizon 40 (`P6`)?
- `dead_work`: two sentences: what does `predict()` compute, who reads its answer, and what would removing it save (p6's `PREDICT_TICKS = 1` line)?
- `horizon_trade`: two sentences: from p6, what did halving the horizon buy and cost, and would you ship it for a race whose tick is 50 ms?

In [ ]:
# The probe record of plan roar-012 (p6): agents/unit-11.py, test-circuit-seed1, 3 laps, without the profiler.
P6 = {
    "step() p50 / p99 (ms), MPC_HORIZON 40": (0.914, 0.995),
    "step() p50 (ms), MPC_HORIZON 20": 0.537,
    "step() p50 (ms), PREDICT_TICKS 1": 0.826,
    "step() p50 (ms), pure pursuit": 0.218,
    "mpc_command share of step(), MPC_HORIZON 40": 0.63,
    "predict share of step()": 0.19,
    "race time (s), MPC_HORIZON 40 / 20": (276.25, 276.35),
}

horizon_path, profiler, horizon_race, stats = None, None, None, None  # TODO
step_p50_ms, step_p99_ms = None, None  # TODO
where_time_goes = "TODO"
dead_work = "TODO"
horizon_trade = "TODO"

## Exercise 8 - The traps, on purpose

**(a) Round trips.**
For each value in `VALUES`, check whether its `repr` reads back as exactly the same float, and whether its 3-decimal text (`f"{v:.3f}"`) does.

**(b) Accumulated steps.**
Start from `A_ACC = 4.0` and add the grid step 0.1 five times, one addition at a time: `accumulated`.
`snapped` is `round(accumulated, 2)`.
Race both as `A_ACC` on `test-circuit`, race seed 0, 1 lap: two races.

**(c) A dead knob.**
Race `K_HEAD = 2.0` on `test-circuit`, race seed 0, 1 lap (one race), and compare it with Exercise 1's shipped race at seed 0.

**(d) Ticks.**
`TIMES` are three race times as racekit returns them.

Define:

- `round_trips`, `three_decimals`: dicts from each value of `VALUES` to `True` or `False`;
- `accumulated`, `snapped`, `same_float` (`accumulated == snapped`);
- `accumulated_race`, `snapped_race`; `same_time`: their times equal; `same_telemetry`: `same_race` of the two;
- `dead_race`, `dead_identical`: `same_race` with the shipped race at seed 0;
- `time_ticks`: a dict from each time of `TIMES` to its whole number of ticks, and `printed`: a dict from each to its text with 2 decimals; `one_tick_apart`: `True` when the first two times are exactly one tick apart in ticks;
- `traps_summary`: three sentences: what goes wrong in (a), (b) and (c) if you are careless, and the one habit that prevents each.

In [ ]:
VALUES = (0.1 + 0.2, 0.54 + 3 * 0.02, 0.546666666606, 2 / 3)
TIMES = (279.65000000000003, 279.6, 96.4)  # s

round_trips, three_decimals = {}, {}  # TODO
accumulated, snapped, same_float = None, None, None  # TODO
accumulated_race, snapped_race = None, None  # TODO
same_time, same_telemetry = None, None  # TODO
dead_race, dead_identical = None, None  # TODO
time_ticks, printed, one_tick_apart = {}, {}, None  # TODO
traps_summary = "TODO"

## Exercise 9 - Challenge: the speed loop and the top speed

The lesson's knob map found three big knobs the crash-aware search never touched: the speed loop's gains `SPEED_KP` and `SPEED_KI`, and the top speed `TARGET_SPEED`.
Search a small box of them by hand: each setting in `SPEED_BOX` (one knob changed), on `test-circuit`, race seeds 0 and 1, 1 lap: ten races.
Compare each with Exercise 1's shipped races at seeds 0 and 1.

Define:

- `speed_knob_races`: a dict from each label of `SPEED_BOX` to its two results;
- `speed_knob_table`: a dict from each label to `(median time minus the shipped median (s), both clean, the larger line_error of the two (m))`;
- `real_gains`: the labels that are clean in both races **and** beat the shipped file by Unit 1's rule;
- `margins`: a dict from each label to `CLEAN_ERROR` minus its larger line error (m): how far it stayed inside the clean rule;
- `fragile`: three sentences: are the gains you found safe to ship?
  Use your `margins`, and the lesson's records: in p2, `SPEED_KP = 0.3` was 9.35 s faster over five 1-lap tracks at a line error of 1.93 m, and in p3 the training-family search found `SPEED_KP = 0.4` and `TARGET_SPEED = 50` **not clean** at 3 laps.

In [ ]:
SPEED_BOX = {
    "SPEED_KP 0.3": {"SPEED_KP": 0.3},
    "SPEED_KP 0.4": {"SPEED_KP": 0.4},
    "SPEED_KI 0.2": {"SPEED_KI": 0.2},
    "TARGET_SPEED 50": {"TARGET_SPEED": 50.0},
    "TARGET_SPEED 55": {"TARGET_SPEED": 55.0},
}

speed_knob_races = {}  # TODO
speed_knob_table = {}  # TODO
real_gains = None  # TODO
margins = {}  # TODO
fragile = "TODO"

## Exercise 10 - Challenge: policy gradient against coordinate search, same budget

The lesson's optional part taught the one-knob policy gradient on a pretend race.
Race it for real, against coordinate search, on **the same race budget**: at most 5 races each.

- **The knob and the race:** `A_ACC`, on `test-circuit`, 1 lap, race seed 0 for every race (one seed: the noise is the same fixed draw for every setting, so this compares the two methods, not settings across seeds).
- **Policy gradient (5 races):** μ = 4.0, σ = 0.25, α = 0.2, `pg_rng = np.random.default_rng(10)`.
  Two updates; each draws N = 2 values `x = round(mu + pg_rng.normal(0.0, sigma), 2)`, races each as `A_ACC`, and updates μ ← μ + α · mean over the draws of (b − Tᵢ) · (xᵢ − μ) / σ², with b the mean of the two times.
  Then race the final μ (rounded to 2 decimals): the policy's answer.
  That is 2 × 2 + 1 = 5 races.
- **Coordinate search (at most 5 races; it stops early when a step no longer wins):** start at 4.0 (Exercise 1's shipped race at seed 0 **is** this race: reuse it, and count it), then 4.5, then 3.5; take the faster of the two if it beats the incumbent by more than `BAND` (0.10 s); then keep stepping (0.5) in that direction while each step beats the incumbent by more than `BAND`, until 5 races have been used.

Define:

- `pg_log`: a list with one dict per update: `"x"` (the two values raced), `"times"`, `"baseline"`, `"mu"` (after the update); `pg_answer`: the final μ rounded, and `pg_answer_time`: its race time;
- `coord_log`: a list of `(A_ACC, race time)` in the order raced (at most 5 entries, the first from Exercise 1); `coord_answer`, `coord_answer_time`: the incumbent at the end and its time;
- `winner`: `"policy gradient"`, `"coordinate search"` or `"tie"` (the answers' times within one tick);
- `honest_comparison`: three sentences: which method gave the better answer on this budget, what would you need to race before believing it is better on other seeds and tracks, and why is a policy gradient built for a different kind of problem than tuning one constant?

In [ ]:
BAND = 0.10  # s: one track, one seed, 1 lap

pg_log, pg_answer, pg_answer_time = [], None, None  # TODO
coord_log, coord_answer, coord_answer_time = [], None, None  # TODO
winner = "TODO"
honest_comparison = "TODO"